# Buku Kerja Hari 6 — Capstone Project

Pendamping **Modul 6** (Capstone). Estimasi 6–10 jam. Prasyarat: Hari 1–5.

Buku kerja ini memandu Anda **merencanakan dan membangun pipeline capstone** secara bertahap pada data latihan kecil, lalu Anda terapkan pola yang sama pada proyek asli Anda. Di dalamnya ada `cek_deliverable(folder)` — pemeriksa otomatis yang bisa Anda pakai pada folder capstone sungguhan sebelum mengumpulkan.

| Bagian | Isi |
|---|---|
| A | Memilih jalur & topik (lembar perencanaan) |
| B | Pertanyaan analisis & struktur folder (L1–L2) |
| C | Pipeline: clean → analyze → visualize → insight (L3–L6) |
| D | README, `main()`, dan pemeriksaan deliverable (L7) |
| E | Pola khusus Jalur B (API + cache) dan Jalur C (arXiv) (L8–L9) |
| F | Self-review & recap |

> **Prinsip:** lebih baik scope kecil yang rapi daripada ide besar yang setengah jadi. Setiap grafik dan tabel harus menjawab pertanyaan analisis.

> **Cara memakai:** jalankan sel dari atas ke bawah (`Shift+Enter`). Pada setiap latihan, ganti baris `# TULIS KODE ANDA DI SINI` dengan jawaban Anda, lalu jalankan sel **Periksa jawaban** di bawahnya. Buka *Petunjuk* bila buntu. Solusi lengkap ada di notebook `solusi/` — buka hanya setelah mencoba sendiri.

In [ ]:
# @title Jalankan sel ini lebih dulu (alat bantu pemeriksa jawaban)
from contextlib import contextmanager
from unittest.mock import patch

STRICT = False
_hasil = {}

@contextmanager
def periksa(kode):
    """Dipakai di sel 'Periksa jawaban': gagal -> pesan petunjuk; lolos -> tanda centang."""
    try:
        yield
        _hasil[kode] = True
        print(f"✅ {kode} benar!")
    except AssertionError as e:
        _hasil[kode] = False
        print(f"❌ {kode} belum tepat: {e}")
        if STRICT: raise
    except Exception as e:
        _hasil[kode] = False
        print(f"❌ {kode} belum bisa dijalankan: {type(e).__name__}: {e}")
        if STRICT: raise

@contextmanager
def masukan(*jawaban):
    """Simulasi input(): with masukan('Ani', '80'): ... -> input() mengembalikan 'Ani' lalu '80'."""
    with patch("builtins.input", side_effect=list(jawaban)):
        yield

def ringkasan(semua):
    """Cetak rekap latihan: ✅ benar, ❌ salah, ⏳ belum dicoba."""
    benar = 0
    for kode in semua:
        s = _hasil.get(kode)
        tanda = "⏳ belum dicoba" if s is None else ("✅ benar" if s else "❌ perlu diperbaiki")
        benar += bool(s)
        print(f"{kode:<6} {tanda}")
    print(f"\nSkor: {benar} dari {len(semua)} latihan benar")

import re, shutil
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
WORK = Path("latihan_hari6")
WORK.mkdir(exist_ok=True)

# Data latihan (FIKTIF): nilai mahasiswa dengan masalah data yang disengaja
_rng = np.random.default_rng(3)
data_latihan = pd.DataFrame({
    "nama": [f" Mhs{i:02d} " if i % 7 == 0 else f"Mhs{i:02d}" for i in range(1, 41)],
    "gender": _rng.choice(["P", "L"], 40),
    "prodi": _rng.choice(["IF", "SI", "TE"], 40),
    "jam_belajar": _rng.integers(2, 20, 40).astype(float),
})
data_latihan["nilai_uts"] = (45 + 2.2 * data_latihan["jam_belajar"] + _rng.normal(0, 6, 40)).round(1)
data_latihan["nilai_uas"] = (48 + 2.0 * data_latihan["jam_belajar"] + _rng.normal(0, 6, 40)).round(1)
data_latihan.loc[[3, 11, 25], "nilai_uts"] = np.nan            # missing sengaja
data_latihan = pd.concat([data_latihan, data_latihan.iloc[[0, 5]]], ignore_index=True)   # duplikat sengaja

def cek_deliverable(root):
    """Periksa deliverable capstone pada folder `root`. Mengembalikan daftar masalah ([] = lengkap)."""
    root = Path(root)
    masalah = []
    if not (root / "README.md").exists():
        masalah.append("README.md tidak ada")
    else:
        teks = (root / "README.md").read_text(encoding="utf-8")
        judul = re.findall(r"^#{1,3} \d+\.", teks, flags=re.M)
        if len(judul) < 9:
            masalah.append(f"README harus punya >= 9 bagian bernomor (template), ditemukan {len(judul)}")
    if not (root / "requirements.txt").exists():
        masalah.append("requirements.txt tidak ada")
    if not list((root / "data" / "processed").glob("*.csv")):
        masalah.append("data/processed/ belum berisi CSV bersih")
    figs = list((root / "figures").glob("*.png")) if (root / "figures").exists() else []
    if len(figs) < 3:
        masalah.append(f"minimal 3 figure PNG di figures/, ditemukan {len(figs)}")
    ins = root / "output" / "insight.txt"
    if not ins.exists():
        masalah.append("output/insight.txt tidak ada")
    else:
        isi = ins.read_text(encoding="utf-8")
        butir = re.findall(r"^\d+\. ", isi, flags=re.M)
        if len(butir) < 5:
            masalah.append(f"insight minimal 5 butir bernomor, ditemukan {len(butir)}")
        if "Batasan" not in isi:
            masalah.append("insight.txt harus memuat bagian 'Batasan:'")
    return masalah


## A. Memilih Jalur dan Topik (Lembar Perencanaan)

Pilih **satu** jalur. Isi tabel di sel Markdown berikutnya (klik dua kali untuk mengedit).

| Jalur | Cocok bila | Risiko utama |
|---|---|---|
| **A — EDA dataset publik** | ingin hasil stabil; koneksi tidak andal | cleaning/insight dangkal |
| **B — API → analisis → visualisasi** | ingin data segar & pipeline nyata | API gagal/berubah |
| **C — arXiv / literatur** | tema riset/skripsi | metadata saja, bias query |

**Kriteria topik sehat:** data bisa diakses dalam ≤ 1 jam · pertanyaan jelas · hasil bisa divisualisasikan · bisa dijelaskan ke orang awam dalam 2 menit.

### Lembar perencanaan Anda

| Butir | Isi Anda |
|---|---|
| Jalur yang dipilih | … |
| Topik / judul sementara | … |
| Sumber data & cara mengaksesnya | … |
| 2–4 pertanyaan analisis (tulis juga di L1) | … |
| Risiko terbesar & rencana cadangan | … |

**Alokasi waktu yang disarankan (±180 menit coding):** muat+inspeksi 20 · cleaning 25 · analisis 40 · visualisasi 45 · insight+export 25 · buffer 25.

## B. Pertanyaan Analisis dan Struktur Folder

**Pertanyaan analisis ditulis lebih dulu** — bukan setelah plot jadi. Pertanyaan yang baik dapat dijawab dengan data, spesifik, dan diakhiri tanda tanya.

### Latihan L1 — Menulis pertanyaan analisis

Isi list `pertanyaan` dengan **2–4 pertanyaan** untuk topik capstone Anda (atau contoh: data nilai mahasiswa). Setiap pertanyaan berupa string yang diakhiri `?` dan minimal 20 karakter.

In [ ]:
pertanyaan = [
    # TULIS KODE ANDA DI SINI (2-4 pertanyaan)
]

In [ ]:
# Periksa jawaban
with periksa('L1'):
    assert isinstance(pertanyaan, list) and 2 <= len(pertanyaan) <= 4, f"jumlah pertanyaan harus 2–4, saat ini {len(pertanyaan) if isinstance(pertanyaan, list) else '?'}"
    for q in pertanyaan:
        assert isinstance(q, str) and q.strip().endswith("?"), f"pertanyaan harus diakhiri '?': {q!r}"
        assert len(q.strip()) >= 20, f"pertanyaan terlalu pendek: {q!r}"

### Latihan L2 — Membuat struktur folder proyek

Buat `buat_struktur(root)` yang membuat folder `data/raw`, `data/processed`, `src`, `figures`, `output` di bawah `root` (`pathlib`, `mkdir(parents=True, exist_ok=True)`) dan file kosong `requirements.txt` berisi baris `pandas`, `matplotlib`, `seaborn`. Kembalikan `Path(root)`.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
UNTUK setiap sub di [data/raw, data/processed, src, figures, output]:
    (root / sub).mkdir(parents=True, exist_ok=True)
tulis requirements.txt; RETURN root
```

</details>

In [ ]:
def buat_struktur(root):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('L2'):
    r = buat_struktur(WORK / "uji_struktur")
    for sub in ("data/raw", "data/processed", "src", "figures", "output"):
        assert (r / sub).is_dir(), f"folder {sub} belum dibuat"
    assert (r / "requirements.txt").read_text(encoding="utf-8").split() == ["pandas", "matplotlib", "seaborn"]
    buat_struktur(WORK / "uji_struktur")  # aman dipanggil dua kali (exist_ok)

## C. Pipeline: clean → analyze → visualize → insight

Pada skeleton capstone, alurnya `load → clean → analyze → visualize → tulis_insight → main`. Berikut `data_latihan` (fiktif) yang sengaja kotor: spasi di nama, nilai UTS kosong, dan baris duplikat. Perhatikan kondisi awalnya:

In [ ]:
print(data_latihan.shape)
print(data_latihan.isna().sum().to_dict(), "| duplikat:", int(data_latihan.duplicated().sum()))
data_latihan.head()

### Latihan L3 — clean(df)

Buat `clean(df)` yang mengembalikan **salinan** bersih: (1) `strip` spasi pada `nama`; (2) isi `nilai_uts` kosong dengan **median** kolom itu; (3) buang baris duplikat (lakukan **setelah** langkah 1–2); (4) tambah `rata_nilai` = rata-rata `nilai_uts` dan `nilai_uas`; (5) tambah `lulus` (`True` bila `rata_nilai >= 70`); (6) reset index.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
d ← salinan df
nama → strip ; nilai_uts kosong → median
drop_duplicates() (SETELAH strip, agar ' Mhs07 ' == 'Mhs07')
rata_nilai ← (uts + uas) / 2 ; lulus ← rata_nilai >= 70 ; reset_index(drop=True)
```

</details>

In [ ]:
def clean(df):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('L3'):
    asli = data_latihan.copy()
    b = clean(data_latihan)
    assert data_latihan.equals(asli), "df asli tidak boleh diubah (pakai salinan)"
    assert b["nilai_uts"].isna().sum() == 0, "nilai_uts masih ada yang kosong"
    assert b["nama"].str.strip().eq(b["nama"]).all(), "spasi pada nama belum dibuang"
    assert not b.duplicated().any(), "masih ada baris duplikat"
    assert len(b) == len(data_latihan) - 2, f"setelah dedup harus {len(data_latihan) - 2} baris, bukan {len(b)}"
    assert np.allclose(b["rata_nilai"], (b["nilai_uts"] + b["nilai_uas"]) / 2)
    assert (b["lulus"] == (b["rata_nilai"] >= 70)).all() and b["lulus"].dtype == bool
    assert b.index.tolist() == list(range(len(b)))

### Latihan L4 — analyze(df)

Buat `analyze(df)` (df hasil `clean`) yang mengembalikan dict berisi: `per_prodi` (Series rata-rata `rata_nilai` per `prodi`), `per_gender` (Series rata-rata `rata_nilai` per `gender`), `lulus_per_prodi` (Series **proporsi** lulus per prodi), dan `korelasi_jam_nilai` (korelasi `jam_belajar` dengan `rata_nilai`, float).

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
per_prodi ← groupby("prodi")["rata_nilai"].mean()
lulus_per_prodi ← groupby("prodi")["lulus"].mean()  (rata-rata boolean = proporsi)
korelasi ← df["jam_belajar"].corr(df["rata_nilai"])
```

</details>

In [ ]:
def analyze(df):
    hasil = {}
    # TULIS KODE ANDA DI SINI
    return hasil

In [ ]:
# Periksa jawaban
with periksa('L4'):
    b = clean(data_latihan)
    h = analyze(b)
    for k in ("per_prodi", "per_gender", "lulus_per_prodi", "korelasi_jam_nilai"):
        assert k in h, f"kunci '{k}' tidak ada di hasil"
    assert sorted(h["per_prodi"].index) == sorted(b["prodi"].unique())
    assert np.allclose(h["per_prodi"], b.groupby("prodi")["rata_nilai"].mean().reindex(h["per_prodi"].index))
    assert ((h["lulus_per_prodi"] >= 0) & (h["lulus_per_prodi"] <= 1)).all(), "lulus_per_prodi harus proporsi 0–1"
    assert isinstance(h["korelasi_jam_nilai"], float) and 0.5 < h["korelasi_jam_nilai"] <= 1, f"korelasi: {h['korelasi_jam_nilai']}"

### Latihan L5 — visualize(df, hasil, fig_dir)

Buat `visualize(df, hasil, fig_dir)` yang menyimpan **minimal 3** grafik PNG (judul + label sumbu pada tiap grafik) ke folder `fig_dir`: `01_hist_nilai.png` (histogram `rata_nilai`), `02_prodi.png` (bar `hasil["per_prodi"]`), dan `03_jam_vs_nilai.png` (scatter `jam_belajar` vs `rata_nilai`). Buat folder bila belum ada, tutup figure setelah disimpan (`plt.close`), dan kembalikan list nama file yang dibuat.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
fig_dir.mkdir(...)
UNTUK setiap grafik: fig, ax ← subplots; gambar; set_title/xlabel/ylabel; savefig(fig_dir / nama); plt.close(fig)
```

</details>

In [ ]:
def visualize(df, hasil, fig_dir):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('L5'):
    b = clean(data_latihan); h = analyze(b)
    fd = WORK / "uji_figur"
    shutil.rmtree(fd, ignore_errors=True)
    nama = visualize(b, h, fd)
    assert sorted(nama) == ["01_hist_nilai.png", "02_prodi.png", "03_jam_vs_nilai.png"], f"nama file: {nama}"
    for n in nama:
        assert (fd / n).exists() and (fd / n).stat().st_size > 1000, f"{n} belum tersimpan/ kosong"
    assert not plt.get_fignums(), "tutup semua figure dengan plt.close(fig) setelah disimpan"

### Latihan L6 — tulis_insight(hasil, path)

Buat `tulis_insight(hasil, path)` yang menulis file teks berisi **minimal 5 butir bernomor** (`1. ...`, `2. ...`) — setiap butir memuat **angka** dari `hasil` — diikuti baris `Batasan:` dan **minimal 2 butir batasan** (diawali `- `). Buat folder induk `path` bila belum ada.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
ambil angka dari hasil (idxmax(), max(), korelasi, ...) dan masukkan ke f-string
tulis: 5+ butir bernomor, baris kosong, "Batasan:", 2+ butir "- ..." 
```

</details>

In [ ]:
def tulis_insight(hasil, path):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('L6'):
    b = clean(data_latihan); h = analyze(b)
    p = WORK / "uji_insight" / "insight.txt"
    shutil.rmtree(p.parent, ignore_errors=True)
    tulis_insight(h, p)
    isi = p.read_text(encoding="utf-8")
    butir = re.findall(r"^(\d+)\. (.+)$", isi, flags=re.M)
    assert len(butir) >= 5, f"butir bernomor: {len(butir)} (minimal 5)"
    assert all(re.search(r"\d", teks) for _, teks in butir), "setiap butir insight harus memuat angka dari data"
    assert "Batasan:" in isi, "tambahkan baris 'Batasan:'"
    bagian_batasan = isi.split("Batasan:")[1]
    assert len(re.findall(r"^- ", bagian_batasan, flags=re.M)) >= 2, "minimal 2 butir batasan diawali '- '"

## D. README, `main()`, dan Pemeriksaan Deliverable

Template README capstone punya 10 bagian: (1) latar belakang, (2) pertanyaan analisis, (3) data, (4) metode, (5) hasil utama, (6) visualisasi, (7) batasan, (8) kesimpulan & kerja lanjutan, (9) cara menjalankan, (10) struktur folder.

### Latihan L7 — README dan main() — menyatukan pipeline

Buat `tulis_readme(root, judul, pertanyaan)` yang menulis `README.md` dengan judul `# <judul>` dan **10 bagian bernomor** `## 1. ...` sampai `## 10. ...` (bagian 2 memuat daftar `pertanyaan`). Lalu buat `main(root)` yang: `buat_struktur(root)` → `clean(data_latihan)` → simpan ke `data/processed/data_bersih.csv` → `analyze` → `visualize` ke `figures/` → `tulis_insight` ke `output/insight.txt` → `tulis_readme`. Kembalikan hasil `cek_deliverable(root)` (harus `[]`).

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
tulis_readme: judul + 10 bagian "## n. nama"; bagian 2 → daftar pertanyaan
main: buat_struktur → clean → to_csv → analyze → visualize → tulis_insight → tulis_readme → RETURN cek_deliverable(root)
```

</details>

In [ ]:
def tulis_readme(root, judul, pertanyaan):
    # TULIS KODE ANDA DI SINI
    pass

def main(root):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('L7'):
    root = WORK / "capstone_uji"
    shutil.rmtree(root, ignore_errors=True)
    masalah = main(root)
    assert masalah == [], f"deliverable belum lengkap: {masalah}"
    teks = (root / "README.md").read_text(encoding="utf-8")
    assert teks.startswith("# Capstone Contoh"), "baris pertama README harus judul '# ...'"
    assert len(re.findall(r"^## \d+\. ", teks, flags=re.M)) == 10, "README harus punya 10 bagian bernomor '## n. ...'"
    assert all(q in teks for q in pertanyaan), "bagian 2 harus memuat semua pertanyaan dari L1"
    assert cek_deliverable(WORK / "tidak_ada") != [], "cek_deliverable pada folder kosong harus melaporkan masalah"

### Memakai pemeriksa pada proyek asli Anda

Setelah capstone Anda selesai, arahkan `cek_deliverable` ke folder proyek Anda sendiri. Ganti path contoh di bawah, lalu hapus tanda `#`.

In [ ]:
# masalah = cek_deliverable("../capstone_nama_anda")
# print("Lengkap ✅" if not masalah else "\n".join("❌ " + m for m in masalah))
print("Contoh hasil pada folder latihan Anda:", cek_deliverable(WORK / "capstone_uji") or "Lengkap ✅")

## E. Pola Khusus Jalur B dan Jalur C

### Latihan L8 — Jalur B — ambil data dengan cache dan fallback

Buat `ambil_dengan_cache(url, cache_path, timeout=5)`: GET `url` → simpan JSON mentah ke `cache_path` (agar tidak perlu mengulang) → kembalikan `(data, "api")`. Jika request gagal (`RequestException`/`ValueError`) dan cache **ada**, kembalikan `(data_dari_cache, "cache")`; jika cache juga tidak ada kembalikan `(None, "gagal")`.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
COBA: GET → raise_for_status → data ← json ; simpan ke cache ; RETURN (data, "api")
TANGKAP RequestException/ValueError:
    JIKA cache ada → RETURN (isi cache, "cache")
    SELAIN ITU → RETURN (None, "gagal")
```

</details>

In [ ]:
import json
import requests

def ambil_dengan_cache(url, cache_path, timeout=5):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('L8'):
    import json as _json, threading
    from http.server import BaseHTTPRequestHandler, HTTPServer

    class _H(BaseHTTPRequestHandler):
        def log_message(self, *a): pass
        def do_GET(self):
            self.send_response(200); self.send_header("Content-Type", "application/json"); self.end_headers()
            self.wfile.write(_json.dumps({"suhu": [30.1, 31.4]}).encode())

    srv = HTTPServer(("127.0.0.1", 0), _H)
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    url_ok = f"http://127.0.0.1:{srv.server_address[1]}/data"
    cache = WORK / "cache_uji" / "mentah.json"
    shutil.rmtree(cache.parent, ignore_errors=True)
    try:
        data, asal = ambil_dengan_cache(url_ok, cache)
        assert (data, asal) == ({"suhu": [30.1, 31.4]}, "api"), f"hasil: {(data, asal)}"
        assert cache.exists(), "data mentah harus disimpan ke cache_path pada request pertama yang berhasil"
    finally:
        srv.shutdown()
    data2, asal2 = ambil_dengan_cache("http://127.0.0.1:9/mati", cache, timeout=1)
    assert (data2, asal2) == ({"suhu": [30.1, 31.4]}, "cache"), f"saat API mati harus memakai cache: {(data2, asal2)}"
    assert ambil_dengan_cache("http://127.0.0.1:9/mati", WORK / "tidak_ada.json", timeout=1) == (None, "gagal")

### Latihan L9 — Jalur C — parameter paginasi arXiv

Buat `params_arxiv(query, halaman, per_halaman=10)` yang mengembalikan dict parameter API arXiv: `search_query`, `start` (indeks mulai: `(halaman-1) × per_halaman`; halaman dimulai dari 1), `max_results`, `sortBy="submittedDate"`, `sortOrder="descending"`. Lemparkan `ValueError` bila `halaman < 1`. Lalu `total_halaman(n_paper, per_halaman=10)` → jumlah halaman yang dibutuhkan (pembulatan ke atas).

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
start ← (halaman - 1) × per_halaman
total_halaman ← pembagian dibulatkan ke atas (-(-n // per))
```

</details>

In [ ]:
def params_arxiv(query, halaman, per_halaman=10):
    # TULIS KODE ANDA DI SINI
    pass

def total_halaman(n_paper, per_halaman=10):
    # TULIS KODE ANDA DI SINI
    pass

In [ ]:
# Periksa jawaban
with periksa('L9'):
    p = params_arxiv('all:"deep learning"', 3, 10)
    assert p == {"search_query": 'all:"deep learning"', "start": 20, "max_results": 10, "sortBy": "submittedDate", "sortOrder": "descending"}, f"params: {p}"
    assert params_arxiv("x", 1)["start"] == 0
    try:
        params_arxiv("x", 0)
    except ValueError:
        pass
    else:
        raise AssertionError("halaman < 1 harus melempar ValueError")
    assert total_halaman(25) == 3 and total_halaman(30) == 3 and total_halaman(31) == 4 and total_halaman(0) == 0
    # Pengingat etika: beri jeda (mis. time.sleep(3)) antar halaman saat memanggil API arXiv sungguhan.

## F. Self-Review dan Recap

**Checklist kualitas (centang sebelum mengumpulkan)**

- [ ] Pertanyaan analisis tertulis — bukan hanya setelah plot jadi
- [ ] Kode berjalan dari awal tanpa error di mesin saya (restart kernel → Run All)
- [ ] Setiap figure punya judul, label, dan disebut di insight
- [ ] Missing values & tipe data ditangani secara sadar
- [ ] Tidak ada klaim berlebihan di luar jangkauan data (ada bagian **Batasan**)
- [ ] README bisa diikuti orang lain dalam ~10 menit
- [ ] `requirements.txt` lengkap · `cek_deliverable(...)` melaporkan `Lengkap ✅`

**Recap**

1. Bagian mana dari pipeline yang paling memakan waktu? Apa yang akan Anda lakukan berbeda?
2. Insight mana yang paling kuat didukung data, dan mana yang paling rapuh? Mengapa?
3. Langkah belajar berikutnya (Git/GitHub, SQL, pengujian, machine learning): mana yang Anda pilih dan alasannya?

> *Jawaban Anda:* …

## Rekap hasil latihan

Jalankan sel di bawah setelah mengerjakan latihan.

In [ ]:
ringkasan(['L1', 'L2', 'L3', 'L4', 'L5', 'L6', 'L7', 'L8', 'L9'])